<a href="https://colab.research.google.com/github/lcandau/histopathology-clip-lab/blob/exp_10_dinov2_backbone/experiments/exp_10_dinov2_backbone/DINOv2_feature_precompute.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# exp_10 -- DINOv2 feature precompute (PyTorch)

This notebook is a **one-time preprocessing step** for exp_10. Reason:
the pinned `transformers==4.46.0` does NOT expose a TensorFlow port of
DINOv2 (`TFAutoModel.from_pretrained('facebook/dinov2-base')` raises
`ValueError: Unrecognized configuration class ... for TFAutoModel`).

The rest of the project's pipeline (training + 3x OOD eval notebooks) is
TF/Keras. Since the DINOv2 backbone is **frozen** in our design, we only
ever need to forward each image through it once. This notebook runs DINOv2
in PyTorch over the 4 datasets we need (LC25000, NCT-CRC, Chaoyang,
LungHist700@20x) and caches the 768-d CLS feature per image to HDF5 on
Google Drive.

Downstream notebooks load these `.h5` files and feed the cached features
straight into the projection head -- no DINOv2 model construction needed
in TF.

**Run order**:
1. This notebook (once).
2. `CLIP_DINOv2_BERT.ipynb` (training; loads `lc25000.h5`).
3. `../exp_07_ood_eval/{NCT,Chaoyang,LungHist700}_OOD_eval.ipynb` with
   `VARIANT = "dinov2_bert_composed"` (loads `nct_crc.h5` /
   `chaoyang.h5` / `lunghist700_20x.h5`).

**Cache invalidation**: delete the `.h5` files under
`/content/drive/MyDrive/clip_histopathology/cache/dinov2/` and re-run this
notebook.

**Compute estimate**: ~25k + ~2k + ~6k + ~0.4k ~= ~34k images.
DINOv2-base on a Colab T4 ~50 img/s batched -> ~10-15 min total.

## 0 -- Bootstrap

In [1]:
# --- Cell 0: bootstrap ---
# PyTorch-only: we intentionally do NOT set KERAS_BACKEND or import TensorFlow.
import os, sys, subprocess

REPO_URL = "https://github.com/lcandau/histopathology-clip-lab.git"
REPO_DIR = "/content/histopathology-clip-lab"
BRANCH   = "exp_10_dinov2_backbone"
IN_COLAB = "google.colab" in sys.modules


def _clone_with_fallback(branch):
    try:
        subprocess.run(
            ["git", "clone", "-b", branch, REPO_URL, REPO_DIR],
            check=True, capture_output=True,
        )
        print(f"Cloned branch {branch!r}")
        return branch
    except subprocess.CalledProcessError:
        print(f"Branch {branch!r} not found; cloning main")
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
        return "main"


if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        active = _clone_with_fallback(BRANCH)
    else:
        fetch = subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], capture_output=True)
        if fetch.returncode == 0:
            subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
            subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
            active = BRANCH
        else:
            subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", "main"], check=True)
            subprocess.run(["git", "-C", REPO_DIR, "checkout", "main"], check=True)
            subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)
            active = "main"
    print(f"Active branch: {active}")
    # PyTorch + transformers + h5py + Pillow + tqdm + kagglehub.
    # We DO NOT install TF here -- this notebook is PyTorch only.
    subprocess.run([
        "pip", "install", "-q",
        "transformers==4.46.0", "h5py", "tqdm",
        "kagglehub", "Pillow", "numpy",
    ], check=True)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
else:
    LOCAL_REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
    if LOCAL_REPO_DIR not in sys.path:
        sys.path.insert(0, LOCAL_REPO_DIR)

print("In Colab:", IN_COLAB)
print("sys.path[0]:", sys.path[0])

Cloned branch 'exp_10_dinov2_backbone'
Active branch: exp_10_dinov2_backbone
Mounted at /content/drive
In Colab: True
sys.path[0]: /content/histopathology-clip-lab


## 1 -- Imports

In [2]:
# --- Cell 1: imports ---
import json
import re
from pathlib import Path

import numpy as np
import h5py
import torch
from PIL import Image
from tqdm.auto import tqdm

from transformers import AutoModel, AutoImageProcessor

from src.data.lc25000 import (
    CLASS_INFO, NUM_CLASSES, CLASS_NAMES, INDEX_TO_NAME, ID_TO_INDEX,
    discover_records,
)
from src.utils.paths import drive_root, results_dir

print(f"torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  device: {torch.cuda.get_device_name(0)}")

torch: 2.11.0+cu128
CUDA available: True
  device: Tesla T4


## 2 -- Configuration

The cache lives at `/content/drive/MyDrive/clip_histopathology/cache/dinov2/`
(under `drive_root()` if available; falls back to a local tmp path).

Per-image cache key: the **POSIX relative path under the dataset root**
(e.g. `lung_image_sets/lung_n/lungn1.jpeg` for LC25000;
`NORM/CRC-VAL-...tif` for NCT; `train/535940-IMG...JPG` for Chaoyang;
the bare basename for LungHist700 since it's flat). The downstream notebooks
rebuild this same key from their own dataset roots, so the cache is portable
as long as the **layout under the dataset root** is unchanged.

In [3]:
# --- Cell 2: config ---
DINOV2_MODEL_ID = "facebook/dinov2-base"
IMG_SIZE   = 224
BATCH_SIZE = 32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if IN_COLAB:
    CACHE_ROOT = Path("/content/drive/MyDrive/clip_histopathology/cache/dinov2")
else:
    # Local fallback for sanity testing -- not where Colab actually writes.
    CACHE_ROOT = Path("/tmp/clip_histopathology/cache/dinov2")
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

DATASET_TAGS = ("lc25000", "nct_crc", "chaoyang", "lunghist700_20x")
CACHE_PATHS = {tag: CACHE_ROOT / f"{tag}.h5" for tag in DATASET_TAGS}

print(f"Model:       {DINOV2_MODEL_ID}")
print(f"Image size:  {IMG_SIZE}")
print(f"Batch size:  {BATCH_SIZE}")
print(f"Device:      {DEVICE}")
print(f"Cache root:  {CACHE_ROOT}")
for tag, p in CACHE_PATHS.items():
    exists = " (exists)" if p.exists() else ""
    print(f"  {tag:<18s} -> {p}{exists}")

Model:       facebook/dinov2-base
Image size:  224
Batch size:  32
Device:      cuda
Cache root:  /content/drive/MyDrive/clip_histopathology/cache/dinov2
  lc25000            -> /content/drive/MyDrive/clip_histopathology/cache/dinov2/lc25000.h5 (exists)
  nct_crc            -> /content/drive/MyDrive/clip_histopathology/cache/dinov2/nct_crc.h5 (exists)
  chaoyang           -> /content/drive/MyDrive/clip_histopathology/cache/dinov2/chaoyang.h5 (exists)
  lunghist700_20x    -> /content/drive/MyDrive/clip_histopathology/cache/dinov2/lunghist700_20x.h5 (exists)


## 3 -- Load DINOv2 + image processor

In [4]:
# --- Cell 3: load model ---
# AutoImageProcessor handles resize to 224x224 + RGB float + ImageNet mean/std.
# We rely on it rather than rolling our own -- it matches the official DINOv2
# preprocessing exactly.
processor = AutoImageProcessor.from_pretrained(DINOV2_MODEL_ID)
model = AutoModel.from_pretrained(DINOV2_MODEL_ID).to(DEVICE).eval()
for p in model.parameters():
    p.requires_grad_(False)

VISION_HIDDEN = int(model.config.hidden_size)  # 768 for ViT-B/14
print(f"DINOv2 vision hidden size: {VISION_HIDDEN}")
print(f"DINOv2 patch size:         {model.config.patch_size}")
print(f"Processor crop / size:     {processor.crop_size if hasattr(processor, 'crop_size') else processor.size}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

DINOv2 vision hidden size: 768
DINOv2 patch size:         14
Processor crop / size:     {'height': 224, 'width': 224}


## 4 -- Shared encode + write helpers

One function batches images through DINOv2 and returns (N, 768) float32
features. A second writes the cache HDF5 with `features`, `paths`, and
model/config attributes.

In [5]:
# --- Cell 4: encode + write helpers ---
@torch.no_grad()
def encode_paths_to_features(paths, *, batch_size=BATCH_SIZE, desc="dinov2"):
    """Forward each image through DINOv2; return (N, 768) float32 CLS features.

    Uses AutoImageProcessor for resize + ImageNet-normalised tensors. The CLS
    token at position 0 of last_hidden_state is the convention recommended by
    DINOv2's authors for downstream classification heads.
    """
    feats = np.zeros((len(paths), VISION_HIDDEN), dtype=np.float32)
    for start in tqdm(range(0, len(paths), batch_size), desc=desc):
        batch_paths = paths[start:start + batch_size]
        images = [Image.open(p).convert("RGB") for p in batch_paths]
        inputs = processor(images=images, return_tensors="pt")
        pixel_values = inputs["pixel_values"].to(DEVICE, non_blocking=True)
        out = model(pixel_values=pixel_values)
        cls = out.last_hidden_state[:, 0, :]  # (B, 768)
        feats[start:start + cls.shape[0]] = cls.detach().to("cpu", non_blocking=True).numpy()
    return feats


def write_cache(out_path, features, rel_keys, *, extra_attrs=None):
    """Write an HDF5 cache with features + relative-path keys.

    Schema:
      features : (N, 768) float32 -- the CLS token features
      paths    : (N,) S512        -- POSIX relative path under the dataset root
    Attributes:
      model_id : str    -- e.g. "facebook/dinov2-base"
      img_size : int    -- 224
      n        : int    -- N
      key_kind : str    -- "relative_path" (POSIX, lower-case extension preserved as-is)
      plus any extras supplied by the caller (e.g. dataset_tag).
    """
    assert features.dtype == np.float32, features.dtype
    assert features.shape == (len(rel_keys), VISION_HIDDEN), features.shape
    rel_arr = np.array([str(k) for k in rel_keys], dtype="S512")
    out_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = out_path.with_suffix(out_path.suffix + ".tmp")
    with h5py.File(tmp_path, "w") as f:
        f.create_dataset("features", data=features, dtype="float32",
                          compression="gzip", compression_opts=4)
        f.create_dataset("paths", data=rel_arr)
        f.attrs["model_id"] = DINOV2_MODEL_ID
        f.attrs["img_size"] = int(IMG_SIZE)
        f.attrs["n"]        = int(len(rel_keys))
        f.attrs["key_kind"] = "relative_path"
        if extra_attrs:
            for k, v in extra_attrs.items():
                f.attrs[k] = v
    tmp_path.replace(out_path)
    print(f"Wrote {out_path}  ({features.shape[0]} features, {VISION_HIDDEN}-d)")


print("helpers ready")

helpers ready


## 5 -- LC25000 (~25k images)

Same path discovery + dedupe that the training notebooks use. Cache key:
relative POSIX path under `lung_colon_image_set/` (e.g.
`lung_image_sets/lung_n/lungn1.jpeg`).

In [6]:
# --- Cell 5: precompute LC25000 ---
import kagglehub

out_path = CACHE_PATHS["lc25000"]
if out_path.exists():
    print(f"LC25000 cache already at {out_path}; skipping.  (delete file to recompute)")
else:
    print("Downloading LC25000 via kagglehub...")
    dataset_path = kagglehub.dataset_download(
        "andrewmvd/lung-and-colon-cancer-histopathological-images"
    )
    print(f"Dataset root: {dataset_path}")
    lc_root = Path(dataset_path) / "lung_colon_image_set"
    all_paths, all_indices = discover_records(dataset_path, apply_dedupe=True)
    print(f"Discovered {len(all_paths)} LC25000 images (post-dedupe).")
    abs_paths = [Path(p) for p in all_paths]
    rel_keys = [p.relative_to(lc_root).as_posix() for p in abs_paths]
    # Smoke-check uniqueness.
    assert len(set(rel_keys)) == len(rel_keys), "LC25000 rel keys not unique!"
    feats = encode_paths_to_features([str(p) for p in abs_paths], desc="LC25000")
    write_cache(out_path, feats, rel_keys, extra_attrs={"dataset_tag": "lc25000"})

LC25000 cache already at /content/drive/MyDrive/clip_histopathology/cache/dinov2/lc25000.h5; skipping.  (delete file to recompute)


## 6 -- NCT-CRC-HE-7K NORM + TUM (~2k images)

Same Zenodo + Drive-cache pattern as `NCT_OOD_eval.ipynb` -- inline-copied
so this notebook is self-contained. We only need NORM and TUM (the two
classes with LC25000 analogues). Cache key: `{NORM|TUM}/<basename>.tif`.

In [7]:
# --- Cell 6: precompute NCT-CRC (NORM + TUM only) ---
import urllib.request, zipfile, shutil

out_path = CACHE_PATHS["nct_crc"]
if out_path.exists():
    print(f"NCT cache already at {out_path}; skipping.  (delete file to recompute)")
else:
    NCT_URL  = "https://zenodo.org/records/1214456/files/CRC-VAL-HE-7K.zip"
    NCT_NAME = "CRC-VAL-HE-7K"
    DRIVE_OOD_ZIP   = drive_root() / "ood_datasets" / f"{NCT_NAME}.zip"
    LOCAL_OOD_ROOT  = Path("/content/ood_data") if IN_COLAB else Path("/tmp/ood_data")
    LOCAL_OOD_DIR   = LOCAL_OOD_ROOT / NCT_NAME

    def _ensure_nct_zip(path):
        path.parent.mkdir(parents=True, exist_ok=True)
        if path.exists():
            print(f"NCT zip already at {path}")
            return path
        DRIVE_OOD_ZIP.parent.mkdir(parents=True, exist_ok=True)
        if DRIVE_OOD_ZIP.exists():
            print(f"Copying NCT zip from Drive cache {DRIVE_OOD_ZIP} -> {path}")
            shutil.copy(DRIVE_OOD_ZIP, path)
            return path
        print(f"Downloading NCT-CRC from {NCT_URL} ...")
        urllib.request.urlretrieve(NCT_URL, path)
        print(f"Copying download to Drive cache: {DRIVE_OOD_ZIP}")
        shutil.copy(path, DRIVE_OOD_ZIP)
        return path

    LOCAL_OOD_ROOT.mkdir(parents=True, exist_ok=True)
    nct_zip_local = LOCAL_OOD_ROOT / f"{NCT_NAME}.zip"
    _ensure_nct_zip(nct_zip_local)
    if not LOCAL_OOD_DIR.exists() or not any(LOCAL_OOD_DIR.iterdir()):
        print(f"Extracting {nct_zip_local} -> {LOCAL_OOD_ROOT}")
        with zipfile.ZipFile(nct_zip_local) as zf:
            zf.extractall(LOCAL_OOD_ROOT)

    nct_root = LOCAL_OOD_DIR
    NCT_KEEP = ("NORM", "TUM")
    abs_paths = []
    for nct_cls in NCT_KEEP:
        class_dir = nct_root / nct_cls
        if not class_dir.exists():
            raise FileNotFoundError(f"Expected NCT class dir not found: {class_dir}")
        for f in sorted(class_dir.iterdir()):
            if f.suffix.lower() in {".tif", ".tiff", ".png", ".jpg", ".jpeg"}:
                abs_paths.append(f)
    rel_keys = [p.relative_to(nct_root).as_posix() for p in abs_paths]
    assert len(set(rel_keys)) == len(rel_keys), "NCT rel keys not unique!"
    print(f"NCT (NORM + TUM): {len(abs_paths)} images")
    feats = encode_paths_to_features([str(p) for p in abs_paths], desc="NCT")
    write_cache(out_path, feats, rel_keys, extra_attrs={"dataset_tag": "nct_crc"})

NCT cache already at /content/drive/MyDrive/clip_histopathology/cache/dinov2/nct_crc.h5; skipping.  (delete file to recompute)


## 7 -- Chaoyang normal + adenocarcinoma (~4k images)

Mirrors the Chaoyang OOD-eval cell. Cache key: relative POSIX path under
`chaoyang-data/` (e.g. `train/535940-IMG009x022-2.JPG`). Combines train+test
since we never train on Chaoyang.

In [8]:
# --- Cell 7: precompute Chaoyang (normal + adenocarcinoma only) ---
out_path = CACHE_PATHS["chaoyang"]
if out_path.exists():
    print(f"Chaoyang cache already at {out_path}; skipping.  (delete file to recompute)")
else:
    CHAOYANG_DRIVE_DIR = Path("/content/drive/MyDrive/TFM/datasets/Chaoyang") if IN_COLAB \
        else Path(os.environ.get("CHAOYANG_LOCAL_DIR", "/tmp/Chaoyang"))
    CHAOYANG_DATASETS_DIR = CHAOYANG_DRIVE_DIR.parent
    CHAOYANG_DATA_ROOT = CHAOYANG_DRIVE_DIR / "chaoyang-data"

    def _find_chaoyang_zip(search_dir):
        if not search_dir.exists():
            return None
        for pattern in ("chaoyang-data*.zip", "chaoyang*.zip", "Chaoyang*.zip"):
            hits = sorted(search_dir.glob(pattern))
            if hits:
                return hits[0]
        return None

    def _extract_zip(zip_path, dest_dir):
        import zipfile
        dest_dir.mkdir(parents=True, exist_ok=True)
        print(f"Extracting {zip_path.name} ({zip_path.stat().st_size / 1e6:.1f} MB) -> {dest_dir} ...")
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(dest_dir)
        print("  done.")

    if not (CHAOYANG_DATA_ROOT / "test.json").is_file():
        zip_path = _find_chaoyang_zip(CHAOYANG_DATASETS_DIR)
        if zip_path is None:
            raise FileNotFoundError(
                f"Chaoyang dataset not found. Looked at {CHAOYANG_DATA_ROOT} and a zip in {CHAOYANG_DATASETS_DIR}.\n"
                "Follow the same Chaoyang setup as the OOD eval notebook (upload chaoyang-data-*.zip to that Drive folder)."
            )
        _extract_zip(zip_path, CHAOYANG_DRIVE_DIR)

    CHAOYANG_LABEL_NAMES = {0: "normal", 1: "serrated", 2: "adenocarcinoma", 3: "adenoma"}
    CHAOYANG_KEEP = {"normal", "adenocarcinoma"}

    def _load_chaoyang_split(split):
        with open(CHAOYANG_DATA_ROOT / f"{split}.json") as fh:
            entries = json.load(fh)
        out = []
        for e in entries:
            cls = CHAOYANG_LABEL_NAMES[int(e["label"])]
            rel = e["name"]
            out.append((CHAOYANG_DATA_ROOT / rel, cls))
        return out

    all_entries = _load_chaoyang_split("train") + _load_chaoyang_split("test")
    abs_paths = []
    for path, cls in all_entries:
        if cls not in CHAOYANG_KEEP:
            continue
        if not path.is_file():
            continue
        abs_paths.append(path)
    rel_keys = [p.relative_to(CHAOYANG_DATA_ROOT).as_posix() for p in abs_paths]
    assert len(set(rel_keys)) == len(rel_keys), "Chaoyang rel keys not unique!"
    print(f"Chaoyang (normal + adenocarcinoma): {len(abs_paths)} images")
    feats = encode_paths_to_features([str(p) for p in abs_paths], desc="Chaoyang")
    write_cache(out_path, feats, rel_keys, extra_attrs={"dataset_tag": "chaoyang"})

Chaoyang cache already at /content/drive/MyDrive/clip_histopathology/cache/dinov2/chaoyang.h5; skipping.  (delete file to recompute)


## 8 -- LungHist700 @ 20x (~360 images)

Same filename-parsing as the LungHist700 OOD-eval cell -- keeps only 20x
tiles and only the three classes with LC25000 analogues (aca/scc/nor).
Cache key: basename (the LungHist700 root is effectively flat in this
release; we record just the filename to keep the key stable across folder
layouts).

In [9]:
# --- Cell 8: precompute LungHist700 @ 20x ---
out_path = CACHE_PATHS["lunghist700_20x"]
if out_path.exists():
    print(f"LungHist700 cache already at {out_path}; skipping.  (delete file to recompute)")
else:
    LUNGHIST_DRIVE_DIR = Path("/content/drive/MyDrive/TFM/datasets/LungHist700") if IN_COLAB \
        else Path(os.environ.get("LUNGHIST_LOCAL_DIR", "/tmp/LungHist700"))
    LUNGHIST_ROOT = LUNGHIST_DRIVE_DIR

    def _has_images(p):
        if not p.exists():
            return False
        return any(True for _ in p.rglob("*.jpg")) or any(True for _ in p.rglob("*.png"))

    if not _has_images(LUNGHIST_ROOT):
        raise FileNotFoundError(
            f"LungHist700 images not found under {LUNGHIST_ROOT}.\n"
            "Follow the LungHist700_OOD_eval.ipynb setup (extract the rar/zip into that Drive folder)."
        )

    LUNG_CLS_PATTERNS = {"aca": "lung adenocarcinoma",
                         "scc": "lung squamous cell carcinoma",
                         "nor": "benign lung tissue"}
    LUNG_MAG_TARGET = "20x"

    def _parse_lunghist_filename(name):
        stem = Path(name).stem.lower()
        tokens = re.split(r"[_\-\s]+", stem)
        cls = mag = None
        for t in tokens:
            if t in LUNG_CLS_PATTERNS and cls is None:
                cls = t
            if t in {"20x", "40x"} and mag is None:
                mag = t
        return cls, mag

    all_files = sorted([p for p in LUNGHIST_ROOT.rglob("*.jpg")]
                       + [p for p in LUNGHIST_ROOT.rglob("*.png")])
    abs_paths = []
    for f in all_files:
        cls, mag = _parse_lunghist_filename(f.name)
        if cls is None or mag != LUNG_MAG_TARGET:
            continue
        abs_paths.append(f)
    rel_keys = [p.name for p in abs_paths]  # basename only -- LungHist is effectively flat
    assert len(set(rel_keys)) == len(rel_keys), "LungHist700 basenames not unique!"
    print(f"LungHist700 @ {LUNG_MAG_TARGET}: {len(abs_paths)} images")
    feats = encode_paths_to_features([str(p) for p in abs_paths], desc="LungHist700")
    write_cache(out_path, feats, rel_keys,
                extra_attrs={"dataset_tag": "lunghist700_20x", "magnification": "20x"})

LungHist700 cache already at /content/drive/MyDrive/clip_histopathology/cache/dinov2/lunghist700_20x.h5; skipping.  (delete file to recompute)


## 9 -- Verify all 4 caches

In [10]:
# --- Cell 9: verify all 4 caches ---
missing = []
for tag, p in CACHE_PATHS.items():
    if not p.exists():
        missing.append(tag)
        continue
    with h5py.File(p, "r") as f:
        feats = f["features"]
        paths = f["paths"]
        print(f"{tag:<18s}  {p}")
        print(f"  features    : {feats.shape} {feats.dtype}")
        print(f"  paths       : {paths.shape} (S{paths.dtype.itemsize})")
        print(f"  attrs       : { {k: f.attrs[k] for k in f.attrs} }")
        print(f"  example key : {paths[0].decode()!r}")
        print()
if missing:
    print(f"MISSING caches: {missing}")
else:
    print("All 4 caches present.")

lc25000             /content/drive/MyDrive/clip_histopathology/cache/dinov2/lc25000.h5
  features    : (23720, 768) float32
  paths       : (23720,) (S512)
  attrs       : {'dataset_tag': 'lc25000', 'img_size': np.int64(224), 'key_kind': 'relative_path', 'model_id': 'facebook/dinov2-base', 'n': np.int64(23720)}
  example key : 'lung_image_sets/lung_n/lungn1.jpeg'

nct_crc             /content/drive/MyDrive/clip_histopathology/cache/dinov2/nct_crc.h5
  features    : (1974, 768) float32
  paths       : (1974,) (S512)
  attrs       : {'dataset_tag': 'nct_crc', 'img_size': np.int64(224), 'key_kind': 'relative_path', 'model_id': 'facebook/dinov2-base', 'n': np.int64(1974)}
  example key : 'NORM/NORM-TCGA-AASSYQPA.tif'

chaoyang            /content/drive/MyDrive/clip_histopathology/cache/dinov2/chaoyang.h5
  features    : (4060, 768) float32
  paths       : (4060,) (S512)
  attrs       : {'dataset_tag': 'chaoyang', 'img_size': np.int64(224), 'key_kind': 'relative_path', 'model_id': 'facebook